# Text2Cypher - 사용자 질의를 Cypher 쿼리로 변환하기

In [1]:
from dotenv import load_dotenv
from neo4j_graphrag.generation.prompts import Text2CypherTemplate
from neo4j_graphrag.llm import OpenAILLM
import re

load_dotenv()

True

## 1. LLM 정의

In [2]:
llm = OpenAILLM(
    model_name="gpt-4o-mini",
    model_params={
        "temperature": 0,
    }
)

## 2. Text2Cypher 프롬프트 템플릿 확인

Neo4j GraphRAG 패키지에서 제공하는 기본 프롬프트 템플릿

In [3]:
prompt_template = Text2CypherTemplate()
print(prompt_template.template)


Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:
{schema}

Examples (optional):
{examples}

Input:
{query_text}

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:



## 3. Text2Cypher 구현

1. 프롬프트 템플릿에 스키마와 질문을 삽입
2. LLM을 호출하여 Cypher 쿼리 생성
3. 생성된 쿼리를 추출하여 반환


- LLM 응답에서 Cypher 쿼리를 추출하는 함수

In [4]:
def extract_cypher(text: str) -> str:
    """
    LLM 응답에서 Cypher 쿼리를 추출하고 포맷팅합니다.

    이 함수는 두 가지 작업을 수행합니다:
    1. 트리플 백틱(```) 안의 Cypher 코드를 추출
    2. 여러 단어로 된 식별자에 자동으로 백틱 추가:
       - 노드 레이블 (예: ":Data Science" → ":`Data Science`")
       - 속성 키 (예: "first name:" → "`first name`:")
       - 관계 타입 (예: "[:WORKS WITH]" → "[:`WORKS WITH`]")

    Args:
        text (str): Cypher 코드를 포함할 수 있는 원시 텍스트

    Returns:
        str: 올바르게 포맷팅된 Cypher 쿼리
    """
    # 트리플 백틱으로 감싸진 Cypher 코드 추출
    pattern = r"```(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    cypher_query = matches[0] if matches else text

    # 공백이 포함된 노드 레이블에 백틱 추가
    cypher_query = re.sub(
        r":\s*(?!`\s*)(\s*)([a-zA-Z0-9_]+(?:\s+[a-zA-Z0-9_]+)+)(?!\s*`)(\s*)",
        r":`\2`",
        cypher_query,
    )

    # 공백이 포함된 속성 키에 백틱 추가
    cypher_query = re.sub(
        r"([,{]\s*)(?!`)([a-zA-Z0-9_]+(?:\s+[a-zA-Z0-9_]+)+)(?!`)(\s*:)",
        r"\1`\2`\3",
        cypher_query,
    )

    # 공백이 포함된 관계 타입에 백틱 추가
    cypher_query = re.sub(
        r"(\[\s*[a-zA-Z0-9_]*\s*:\s*)(?!`)([a-zA-Z0-9_]+(?:\s+[a-zA-Z0-9_]+)+)(?!`)(\s*(?:\]|-))",
        r"\1`\2`\3",
        cypher_query,
    )

    return cypher_query.strip()

In [5]:
def text_to_cypher(
    question: str,
    schema: str,
    examples: str = "",
) -> dict:
    """
    자연어 질문을 Cypher 쿼리로 변환합니다.

    Args:
        question (str): 사용자의 자연어 질문
        schema (str): Neo4j 그래프 스키마 정보
        examples (str): 예시 질문-쿼리 쌍 (선택사항)

    Returns:
        dict: {
            'question': 원본 질문,
            'prompt': LLM에 전달된 프롬프트,
            'llm_response': LLM의 원시 응답,
            'cypher_query': 추출된 Cypher 쿼리
        }
    """
    # 프롬프트 템플릿 생성
    template = Text2CypherTemplate()

    # 프롬프트 포맷팅
    prompt = template.format(
        schema=schema,
        examples=examples,
        query_text=question
    )

    print("[질문]", question)
    print("\nLLM에 전달하는 프롬프트:")
    print("-" * 80)
    print(prompt)
    print("-" * 80)

    # LLM 호출
    llm_result = llm.invoke(prompt)
    llm_response = llm_result.content

    print("\n[LLM 응답]")
    print("-" * 80)
    print(llm_response)
    print("-" * 80)

    # Cypher 쿼리 추출
    cypher_query = extract_cypher(llm_response)

    print("\n[추출된 Cypher 쿼리]")
    print("-" * 80)
    print(cypher_query)
    print("-" * 80)

    return {
        'question': question,
        'prompt': prompt,
        'llm_response': llm_response,
        'cypher_query': cypher_query
    }

### 예제 1: 영화 관련 질문

In [6]:
# 예시: 영화 데이터베이스 스키마
movie_schema = """
Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
"""

print(movie_schema)


Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)



In [7]:
question1 = "Tom Hanks가 출연한 영화를 모두 찾아줘"

result1 = text_to_cypher(
    question=question1,
    schema=movie_schema
)

[질문] Tom Hanks가 출연한 영화를 모두 찾아줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


Examples (optional):


Input:
Tom Hanks가 출연한 영화를 모두 찾아줘

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:

-------------

In [8]:
# 예제 2: 조건을 포함하는 복잡한 질문
question2 = "2000년 이후에 개봉한 액션 영화 중 평점이 8점 이상인 영화와 감독을 찾아줘"

result2 = text_to_cypher(
    question=question2,
    schema=movie_schema
)

[질문] 2000년 이후에 개봉한 액션 영화 중 평점이 8점 이상인 영화와 감독을 찾아줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


Examples (optional):


Input:
2000년 이후에 개봉한 액션 영화 중 평점이 8점 이상인 영화와 감독을 찾아줘

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your r

In [9]:
# 예제 질문-쿼리 쌍 정의
examples = """
# 배우가 출연한 영화 찾기
Question: Keanu Reeves가 출연한 영화는?
Cypher: MATCH (p:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie) RETURN m.title

# 특정 장르의 영화 찾기
Question: 액션 장르의 영화를 찾아줘
Cypher: MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Action'}) RETURN m.title

# 감독과 영화 찾기
Question: Christopher Nolan이 감독한 영화는?
Cypher: MATCH (p:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie) RETURN m.title
"""
print(examples)


# 배우가 출연한 영화 찾기
Question: Keanu Reeves가 출연한 영화는?
Cypher: MATCH (p:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie) RETURN m.title

# 특정 장르의 영화 찾기
Question: 액션 장르의 영화를 찾아줘
Cypher: MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Action'}) RETURN m.title

# 감독과 영화 찾기
Question: Christopher Nolan이 감독한 영화는?
Cypher: MATCH (p:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie) RETURN m.title



In [10]:
question3 = "Tom Cruise가 출연한 영화를 찾아줘"

result3 = text_to_cypher(
    question=question3,
    schema=movie_schema,
    examples=examples
)

[질문] Tom Cruise가 출연한 영화를 찾아줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Movie {title: STRING, released: INTEGER, tagline: STRING}
- Person {name: STRING, born: INTEGER}
- Genre {name: STRING}

Relationship properties:
- ACTED_IN {roles: LIST}
- DIRECTED {}
- PRODUCED {}
- WROTE {}
- REVIEWED {rating: INTEGER, summary: STRING}
- IN_GENRE {}

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


Examples (optional):

# 배우가 출연한 영화 찾기
Question: Keanu Reeves가 출연한 영화는?
Cypher: MATCH (p:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie) RETURN m.title

# 특정 장르의 영화 찾기
Question: 액션 장르의 영화를 찾아줘
Cypher: MATCH (m:Movie)-[:IN_GENRE]->(g:Genre {name: 'Action'}) RETURN m.tit

### 예제 2: 의료 지식 그래프

In [16]:
# 의료 지식 그래프 스키마 (medical2kg from part2)
medical_schema = """
Node properties:
- Disease {name: STRING, type: STRING}
- Symptom {name: STRING, type: STRING}
- Test {name: STRING, type: STRING}
- Treatment {name: STRING, type: STRING}
- Medication {name: STRING, type: STRING}
- Anatomy {name: STRING, type: STRING}
- Department {name: STRING} : 내과, 산부인과, 응급의학과, 소아과 4가지 진료과를 포함
- Question {qa_id: INTEGER, text: STRING, department: STRING}
- Answer {qa_id: INTEGER, text: STRING, department: STRING}

Relationship properties:
- INDICATES {}
- DIAGNOSES_FOR {}
- TREATS {}
- MENTIONS {}
- ANSWERED_BY {}

The relationships:
(:Symptom)-[:INDICATES]->(:Disease)
(:Test)-[:DIAGNOSES_FOR]->(:Disease)
(:Treatment)-[:TREATS]->(:Disease)
(:Medication)-[:TREATS]->(:Disease)
(:Question)-[:MENTIONS]->(:Disease)
(:Question)-[:MENTIONS]->(:Symptom)
(:Question)-[:MENTIONS]->(:Test)
(:Question)-[:MENTIONS]->(:Treatment)
(:Question)-[:MENTIONS]->(:Medication)
(:Question)-[:MENTIONS]->(:Anatomy)
(:Answer)-[:MENTIONS]->(:Disease)
(:Answer)-[:MENTIONS]->(:Symptom)
(:Answer)-[:MENTIONS]->(:Test)
(:Answer)-[:MENTIONS]->(:Treatment)
(:Answer)-[:MENTIONS]->(:Medication)
(:Answer)-[:MENTIONS]->(:Anatomy)
(:Question)-[:ANSWERED_BY]->(:Answer)
(:Question)-[:BELONGS_TO]->(:Department)
"""
print(medical_schema)


Node properties:
- Disease {name: STRING, type: STRING}
- Symptom {name: STRING, type: STRING}
- Test {name: STRING, type: STRING}
- Treatment {name: STRING, type: STRING}
- Medication {name: STRING, type: STRING}
- Anatomy {name: STRING, type: STRING}
- Department {name: STRING} : 내과, 산부인과, 응급의학과, 소아과 4가지 진료과를 포함
- Question {qa_id: INTEGER, text: STRING, department: STRING}
- Answer {qa_id: INTEGER, text: STRING, department: STRING}

Relationship properties:
- INDICATES {}
- DIAGNOSES_FOR {}
- TREATS {}
- MENTIONS {}
- ANSWERED_BY {}

The relationships:
(:Symptom)-[:INDICATES]->(:Disease)
(:Test)-[:DIAGNOSES_FOR]->(:Disease)
(:Treatment)-[:TREATS]->(:Disease)
(:Medication)-[:TREATS]->(:Disease)
(:Question)-[:MENTIONS]->(:Disease)
(:Question)-[:MENTIONS]->(:Symptom)
(:Question)-[:MENTIONS]->(:Test)
(:Question)-[:MENTIONS]->(:Treatment)
(:Question)-[:MENTIONS]->(:Medication)
(:Question)-[:MENTIONS]->(:Anatomy)
(:Answer)-[:MENTIONS]->(:Disease)
(:Answer)-[:MENTIONS]->(:Symptom)
(:Answer

In [18]:
# 의료 도메인 질문
medical_question = "발열 증상이 발생할 때 복용할 수 있는 약과 치료법을 알려줘"

result4 = text_to_cypher(
    question=medical_question,
    schema=medical_schema
)

[질문] 발열 증상이 발생할 때 복용할 수 있는 약과 치료법을 알려줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Disease {name: STRING, type: STRING}
- Symptom {name: STRING, type: STRING}
- Test {name: STRING, type: STRING}
- Treatment {name: STRING, type: STRING}
- Medication {name: STRING, type: STRING}
- Anatomy {name: STRING, type: STRING}
- Department {name: STRING} : 내과, 산부인과, 응급의학과, 소아과 4가지 진료과를 포함
- Question {qa_id: INTEGER, text: STRING, department: STRING}
- Answer {qa_id: INTEGER, text: STRING, department: STRING}

Relationship properties:
- INDICATES {}
- DIAGNOSES_FOR {}
- TREATS {}
- MENTIONS {}
- ANSWERED_BY {}

The relationships:
(:Symptom)-[:INDICATES]->(:Disease)
(:Test)-[:DIAGNOSES_FOR]->(:Disease)
(:Treatment)-[:TREATS]->(:Disease)
(:Medication)-[:TREATS]->(:Disease)
(:Question)-[:MENTIONS]->(:Disease)
(:Question)-[:MENTIONS

### 예제 3: 법령 지식 그래프

In [19]:
# 법령 지식 그래프 스키마 (law2kg from part2)
law_schema = """
Node properties:
- Law {law_id: STRING, name: STRING, short_name: STRING, category: STRING, promulgation_date: STRING, enforcement_date: STRING, authority: STRING, status: STRING}
- Article {article_id: STRING, number: STRING, title: STRING, content: STRING, law_id: STRING, order: INTEGER}
- Paragraph {paragraph_id: STRING, number: INTEGER, content: STRING, article_id: STRING, order: INTEGER}
- Item {item_id: STRING, number: INTEGER, content: STRING, paragraph_id: STRING, order: INTEGER}
- LegalInterpretation {interpretation_id: STRING, title: STRING, case_number: STRING, answer_date: DATETIME, status: STRING}
- Question {question_id: STRING, text: STRING}
- Answer {answer_id: STRING, text: STRING}
- Reason {reason_id: STRING, text: STRING}
- Organization {name: STRING}

Relationship properties:
- HAS_ARTICLE {order: INTEGER}
- HAS_PARAGRAPH {order: INTEGER}
- HAS_ITEM {order: INTEGER}
- INTERPRETS {}
- CITES {}
- HAS_QUESTION {}
- HAS_ANSWER {}
- ANSWERED_BY {}
- SUPPORTED_BY {}
- REQUESTED {}

The relationships:
(:Law)-[:HAS_ARTICLE]->(:Article)
(:Article)-[:HAS_PARAGRAPH]->(:Paragraph)
(:Paragraph)-[:HAS_ITEM]->(:Item)
(:LegalInterpretation)-[:INTERPRETS]->(:Law)
(:LegalInterpretation)-[:CITES]->(:Article)
(:LegalInterpretation)-[:CITES]->(:Paragraph)
(:LegalInterpretation)-[:CITES]->(:Item)
(:LegalInterpretation)-[:HAS_QUESTION]->(:Question)
(:LegalInterpretation)-[:HAS_ANSWER]->(:Answer)
(:Question)-[:ANSWERED_BY]->(:Answer)
(:Answer)-[:SUPPORTED_BY]->(:Reason)
(:Organization)-[:REQUESTED]->(:LegalInterpretation)
"""
print(law_schema)


Node properties:
- Law {law_id: STRING, name: STRING, short_name: STRING, category: STRING, promulgation_date: STRING, enforcement_date: STRING, authority: STRING, status: STRING}
- Article {article_id: STRING, number: STRING, title: STRING, content: STRING, law_id: STRING, order: INTEGER}
- Paragraph {paragraph_id: STRING, number: INTEGER, content: STRING, article_id: STRING, order: INTEGER}
- Item {item_id: STRING, number: INTEGER, content: STRING, paragraph_id: STRING, order: INTEGER}
- LegalInterpretation {interpretation_id: STRING, title: STRING, case_number: STRING, answer_date: DATETIME, status: STRING}
- Question {question_id: STRING, text: STRING}
- Answer {answer_id: STRING, text: STRING}
- Reason {reason_id: STRING, text: STRING}
- Organization {name: STRING}

Relationship properties:
- HAS_ARTICLE {order: INTEGER}
- HAS_PARAGRAPH {order: INTEGER}
- HAS_ITEM {order: INTEGER}
- INTERPRETS {}
- CITES {}
- HAS_QUESTION {}
- HAS_ANSWER {}
- ANSWERED_BY {}
- SUPPORTED_BY {}
- RE

In [21]:
# 법령 도메인 질문
law_question = "모든 법령의 조-항-목 구조를 나열해줘"

result5 = text_to_cypher(
    question=law_question,
    schema=law_schema
)

[질문] 모든 법령의 조-항-목 구조를 나열해줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Law {law_id: STRING, name: STRING, short_name: STRING, category: STRING, promulgation_date: STRING, enforcement_date: STRING, authority: STRING, status: STRING}
- Article {article_id: STRING, number: STRING, title: STRING, content: STRING, law_id: STRING, order: INTEGER}
- Paragraph {paragraph_id: STRING, number: INTEGER, content: STRING, article_id: STRING, order: INTEGER}
- Item {item_id: STRING, number: INTEGER, content: STRING, paragraph_id: STRING, order: INTEGER}
- LegalInterpretation {interpretation_id: STRING, title: STRING, case_number: STRING, answer_date: DATETIME, status: STRING}
- Question {question_id: STRING, text: STRING}
- Answer {answer_id: STRING, text: STRING}
- Reason {reason_id: STRING, text: STRING}
- Organization {name: ST

### 예제 4: PDF 문서 구조 그래프 

In [26]:
# PDF 문서 구조 그래프 스키마 (pdf2kg from part2)
pdf_schema = """
Node properties:
- Document {title: STRING, pdf_path: STRING, total_pages: INTEGER}
- TOC {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level1 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level2 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level3 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level4 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level5 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Chunk {chunk_id: STRING, toc_id: STRING, content: STRING, text_count: INTEGER, table_count: INTEGER}
- TextElement {element_id: STRING, toc_id: STRING, content: STRING, page: INTEGER} : 텍스트 요소를 저장한 노드
- TableElement {element_id: STRING, toc_id: STRING, page: INTEGER, content: STRING} : 테이블 혹은 표를 마크다운으로 저장한 텍스트를 가짐

Relationship properties:
- HAS_CHILD {}
- HAS_CHUNK {}

The relationships:
(:TOC)-[:HAS_CHILD]->(:TOC)
(:Level1)-[:HAS_CHILD]->(:Level2)
(:Level2)-[:HAS_CHILD]->(:Level3)
(:Level3)-[:HAS_CHILD]->(:Level4)
(:Level4)-[:HAS_CHILD]->(:Level5)
(:TOC)-[:HAS_CHUNK]->(:Chunk)
(:Chunk)-[:HAS_ELEMENT]->(:TextElement)
(:Chunk)-[:HAS_ELEMENT]->(:TableElement)
"""
print(pdf_schema)


Node properties:
- Document {title: STRING, pdf_path: STRING, total_pages: INTEGER}
- TOC {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level1 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level2 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level3 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level4 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level5 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Chunk {chunk_id: STRING, toc_id: STRING, content: STRING, text_count: INTEGER, table_count: INTEGER}
- TextElement {element_id: STRING, toc_id: STRING, content: STRING, page: INTEGER} : 텍스트 요소를 저장한 노드
- TableElement {element_i

In [27]:
# PDF 도메인 질문
pdf_question = "표가 포함된 섹션을 찾아줘"

result6 = text_to_cypher(
    question=pdf_question,
    schema=pdf_schema
)

[질문] 표가 포함된 섹션을 찾아줘

LLM에 전달하는 프롬프트:
--------------------------------------------------------------------------------

Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

Schema:

Node properties:
- Document {title: STRING, pdf_path: STRING, total_pages: INTEGER}
- TOC {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level1 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level2 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level3 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level4 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Level5 {toc_id: STRING, title: STRING, level: INTEGER, parent_id: STRING, is_leaf: BOOLEAN, page_start: INTEGER}
- Chunk {ch

---

## 참고 자료
- [Neo4j GraphRAG Python 문서](https://neo4j.com/docs/neo4j-graphrag-python/current/)
- [Text2CypherRetriever](https://neo4j.com/docs/neo4j-graphrag-python/current/api.html#text2cypherretriever)
- [Text2CypherRetriever 소스코드](https://neo4j.com/docs/neo4j-graphrag-python/current/_modules/neo4j_graphrag/retrievers/text2cypher.html)